- Пока так и осталось не ясным как выявлять когда и янь или инь добавлять период в 15 дней. Каждый год лучше первый расчет проверять на Мингли
- Стартовую точку для часовых расчетов м.б. имеет "двигать" ближе к расчетному году.
- На всякий случай следуем проверять смену сезонов по солнечному календарю.

In [1]:
import MyTools as MT
from importlib import reload
import Variable as V
import sqlite3
import pandas as pd
from datetime import datetime, timedelta, date

import numpy as np
from astropy.time import Time
from astropy.coordinates import get_sun, GeocentricTrueEcliptic
from astropy import units as u

In [ ]:
year_st = 2017
year_en = 2028  # Ограничиваем до 2028 года для избежания проблем с астрономическими вычислениями

## Получаем небесные и земные стволы для каждого года

In [3]:
# формируем цикл Дзя Дзи на 180 лет
def get_jiazi(year, start_year):
    heavenly_stems = ["甲", "乙", "丙", "丁", "戊", "己", "庚", "辛", "壬", "癸"]
    earthly_branches = ["子", "丑", "寅", "卯", "辰", "巳", "午", "未", "申", "酉", "戌", "亥"]

    stem_index = (year - start_year) % 10
    branch_index = (year - start_year) % 12

    return [heavenly_stems[stem_index], stem_index,
            earthly_branches[branch_index], branch_index,
        heavenly_stems[stem_index] + earthly_branches[branch_index]]

start_year = 1864
end_year = year_en
cycle_length = 60
cycle_period = 3 # Верхний период, срдений период и нижний период

tb_nm = 't_cyrcle_jiazi_for_year_01'
# Пересоздаем таблицу Циклов Дзя Дзи для каждого года
MT.drop_table(tb_nm)
sql = f'''CREATE TABLE IF NOT EXISTS {tb_nm}
                  (id INTEGER PRIMARY KEY AUTOINCREMENT,
                   year INTEGER,
                   jiazi TEXT,
                   num_heavenly_stems integer,
                   heavenly_stems text,
                   num_earthly_branches integer,
                   earthly_branches text,
                   cycle integer,
                   year_number integer)'''
MT.gp_execute(sql)
for year in range(start_year, end_year + 1):
    if year - start_year + 1 > 180:
        start_year = start_year + 180
        
    jiazi = get_jiazi(year, start_year)
    cycle_number = (year - start_year) // cycle_length + 1
    year_number = (year - start_year) % cycle_length + 1

    MT.insert_data(tb = tb_nm, column = ['year', 'jiazi', 'num_heavenly_stems', 'heavenly_stems', 'num_earthly_branches', 'earthly_branches', 'cycle', 'year_number'], value = [year
                                                                                            , jiazi[4]
                                                                                            , jiazi[1]+1
                                                                                            , jiazi[0]
                                                                                            , jiazi[3]+1
                                                                                            , jiazi[2]
                                                                                            , cycle_number, year_number])
    
    #print(f"{year}: {jiazi} (Цикл {cycle_number}, Номер расклада =  {year_number})")


print("Цикл Цзя-цзы успешно сгенерирован и сохранен в базе данных SQLite.")

Таблица t_cyrcle_jiazi_for_year_01 успешно удалена
Цикл Цзя-цзы успешно сгенерирован и сохранен в базе данных SQLite.


In [4]:
# проставляем номер расклада в зависимости от того к какому периоду (верхнему, средниму, низкому) соответствует год
tb_name = 't_cyrcle_jiazi_for_year'
sql = """
select vh.*
, case 
	 when vh."cycle" = 1 then spr.high_period		
	 when vh."cycle" = 2 then spr.avg_period
	 when vh."cycle" = 3 then spr.low_period
end  as num_situation
, spr.level_year_for_month

from t_cyrcle_jiazi_for_year_01 vh

left join t_spr_jiazi_for_year spr
on vh.jiazi = spr.jiazi
"""
MT.drop_table(tb_name)
MT.create_table(tb_target = tb_name, sql_script = sql)

Таблица t_cyrcle_jiazi_for_year успешно удалена
Таблица t_cyrcle_jiazi_for_year Успешно создана
Скрипт create_table выполнился за 0 дней 0 часов 0 минут 0 секунд (с 2024-08-23 08:58:59 по 2024-08-23 08:58:59)


## Расчитываем небесный ствол для месяца

In [5]:
# Собираем справочник
year = ["甲", "乙", "丙", "丁", "戊", "己", "庚", "辛", "壬", "癸"]
month = ["丙", "丁", "戊", "己", "庚", "辛", "壬", "癸", "甲", "乙"]
tb_nm = 't_spr_jiazi_for_month'
j = 0
sql = f"""
CREATE TABLE {tb_nm} (
	id INTEGER PRIMARY KEY AUTOINCREMENT,
	jiazi_year TEXT,
	num_mont_cha INTEGER,
	jiazi_month TEXT
)
"""

MT.drop_table(tb_nm)
MT.gp_execute(sql)
for god in year:
    for i in range(1,13):
        j+=1
        MT.insert_data(tb = tb_nm, column = ['jiazi_year', 'num_mont_cha', 'jiazi_month'], value = [god
                                                                                            , i
                                                                                            , month[j%10-1]
                                                                                            ])
        #print (f'Для ствола года {god} месяц {i} - cтвол месяца {month[j%10-1]}')

Таблица t_spr_jiazi_for_month успешно удалена


## Считаем даты и время смены сезонов

In [6]:
tb_nm = 't_spr_sun_dt'
sql = f"""
CREATE TABLE "{tb_nm}" (
	god INTEGER,
	mes INTEGER,
	den INTEGER,
	chas INTEGER,
	minuta INTEGER,
	dt INTEGER,
	deg INTEGER,
    god_cha integer,
    mes_cha integer
);
"""
MT.drop_table(tb_nm)
MT.gp_execute(sql)

def find_sun_crossing(year, degrees):
    # Начало и конец года
    start_time = Time(f'{year}-01-01 00:00:00')
    end_time = Time(f'{year}-12-31 23:59:59')
    
    # Временной интервал для поиска пересечений
    dt = end_time - start_time
    
    # Создаем массив временных точек для проверки положения Солнца
    times = start_time + dt * np.linspace(0, 1, 100000)
    
    # Получаем положение Солнца в каждый из этих моментов
    sun_positions = get_sun(times).transform_to(GeocentricTrueEcliptic())
    
    # Эклиптическая долгота Солнца
    sun_longitudes = sun_positions.lon.wrap_at(360 * u.deg).degree
    
    results = []
    
    for degree in degrees:
        # Найдем время, когда Солнце ближе всего к заданному градусу
        idx = (np.abs(sun_longitudes - degree)).argmin()
        closest_time = times[idx].iso
        closest_degree = sun_longitudes[idx]
        results.append((degree, closest_time, closest_degree))
    
    return results

degrees = [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180, 195, 210, 225, 240, 255, 270, 285, 300, 315, 330, 345]


for year in range(year_st, year_en):
    print(f"Year: {year}")
    crossings = find_sun_crossing(year, degrees)
    for degree, time, actual_degree in crossings:
        dt = datetime.strptime(time, '%Y-%m-%d %H:%M:%S.%f')
        if degree in (315, 330, 345, 0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180, 195, 210, 225, 240, 255, 270):
            god_cha = dt.year 
            mes_cha = dt.month - 1
        elif degree in (285, 300):
            god_cha = dt.year - 1
            mes_cha = 12
        MT.insert_data(tb = tb_nm, column = ['god', 'mes', 'den', 'chas', 'minuta', 'dt', 'deg', 'god_cha', 'mes_cha'], value = [dt.year
                                                                                            , dt.month
                                                                                            , dt.day
                                                                                            , dt.hour
                                                                                            , dt.minute
                                                                                            , time
                                                                                            , degree
                                                                                            , god_cha
                                                                                            , mes_cha
                                                                                            ])
        print(f"Degree: {degree}° - Time: {time}, Actual Degree: {actual_degree:.2f}°")
    print("\n")

Таблица t_spr_sun_dt успешно удалена
Year: 2017
Degree: 0° - Time: 2017-03-20 16:24:43.287, Actual Degree: 0.00°
Degree: 15° - Time: 2017-04-04 20:13:06.423, Actual Degree: 15.00°
Degree: 30° - Time: 2017-04-20 03:26:28.721, Actual Degree: 30.00°
Degree: 45° - Time: 2017-05-05 13:33:18.003, Actual Degree: 45.00°
Degree: 60° - Time: 2017-05-21 02:33:34.268, Actual Degree: 60.00°
Degree: 75° - Time: 2017-06-05 17:45:14.612, Actual Degree: 75.00°
Degree: 90° - Time: 2017-06-21 10:31:31.493, Actual Degree: 90.00°
Degree: 105° - Time: 2017-07-07 03:59:51.279, Actual Degree: 105.00°
Degree: 120° - Time: 2017-07-22 21:28:11.065, Actual Degree: 120.00°
Degree: 135° - Time: 2017-08-07 13:53:26.493, Actual Degree: 135.00°
Degree: 150° - Time: 2017-08-23 04:28:19.295, Actual Degree: 150.00°
Degree: 165° - Time: 2017-09-07 16:46:32.656, Actual Degree: 165.00°
Degree: 180° - Time: 2017-09-23 02:06:03.669, Actual Degree: 180.00°
Degree: 195° - Time: 2017-10-08 08:26:52.336, Actual Degree: 195.00°
De

C:\Users\Дмитрий\AppData\Local\Programs\Python\Python311\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\Дмитрий\AppData\Local\Programs\Python\Python311\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "utctai" yielded 1 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\Дмитрий\AppData\Local\Programs\Python\Python311\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "taiutc" yielded 274 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\Дмитрий\AppData\Local\Programs\Python\Python311\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "utctai" yielded 274 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\Дмитрий\AppData\Local\Programs\Python\Python311\Lib\site

Degree: 0° - Time: 2028-03-20 12:04:46.202, Actual Degree: 0.00°
Degree: 15° - Time: 2028-04-04 15:49:42.839, Actual Degree: 15.00°
Degree: 30° - Time: 2028-04-19 23:05:28.563, Actual Degree: 30.00°
Degree: 45° - Time: 2028-05-05 09:09:53.555, Actual Degree: 45.00°
Degree: 60° - Time: 2028-05-20 22:13:30.271, Actual Degree: 60.00°
Degree: 75° - Time: 2028-06-05 13:23:36.438, Actual Degree: 75.00°
Degree: 90° - Time: 2028-06-21 06:13:50.922, Actual Degree: 90.00°
Degree: 105° - Time: 2028-07-06 23:46:15.222, Actual Degree: 105.00°
Degree: 120° - Time: 2028-07-22 17:08:07.068, Actual Degree: 120.00°
Degree: 135° - Time: 2028-08-07 09:37:16.643, Actual Degree: 135.00°
Degree: 150° - Time: 2028-08-23 00:10:29.220, Actual Degree: 150.00°
Degree: 165° - Time: 2028-09-07 12:26:39.892, Actual Degree: 165.00°
Degree: 180° - Time: 2028-09-22 21:48:55.067, Actual Degree: 180.00°
Degree: 195° - Time: 2028-10-08 04:06:42.292, Actual Degree: 195.00°
Degree: 210° - Time: 2028-10-23 07:09:29.112, Actu

C:\Users\Дмитрий\AppData\Local\Programs\Python\Python311\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "taiutc" yielded 100000 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\Дмитрий\AppData\Local\Programs\Python\Python311\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "utctai" yielded 100000 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\Дмитрий\AppData\Local\Programs\Python\Python311\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 7 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
C:\Users\Дмитрий\AppData\Local\Programs\Python\Python311\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 1 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


Degree: 0° - Time: 2029-03-20 18:09:50.550, Actual Degree: 0.00°
Degree: 15° - Time: 2029-04-04 22:08:44.412, Actual Degree: 15.00°
Degree: 30° - Time: 2029-04-20 05:11:35.984, Actual Degree: 30.00°
Degree: 45° - Time: 2029-05-05 15:28:55.992, Actual Degree: 45.00°
Degree: 60° - Time: 2029-05-21 04:23:56.894, Actual Degree: 60.00°
Degree: 75° - Time: 2029-06-05 19:40:52.602, Actual Degree: 75.00°
Degree: 90° - Time: 2029-06-21 12:21:54.119, Actual Degree: 90.00°
Degree: 105° - Time: 2029-07-07 06:00:44.631, Actual Degree: 105.00°
Degree: 120° - Time: 2029-07-22 23:18:33.691, Actual Degree: 120.00°
Degree: 135° - Time: 2029-08-07 15:43:49.119, Actual Degree: 135.00°
Degree: 150° - Time: 2029-08-23 06:23:57.285, Actual Degree: 150.00°
Degree: 165° - Time: 2029-09-07 18:42:10.645, Actual Degree: 165.00°
Degree: 180° - Time: 2029-09-23 04:01:41.659, Actual Degree: 180.00°
Degree: 195° - Time: 2029-10-08 10:17:14.962, Actual Degree: 195.00°
Degree: 210° - Time: 2029-10-23 13:23:35.192, Actu

In [7]:
tb_nm = 't_bazci_01'
sql = """
select 
    dt
    , god_cha
    , mes_cha
    , deg
FROM t_spr_sun_dt
ORDER BY dt
"""
query = MT.gp_execute(sql_script = sql, rezim = 'list')
df = pd.DataFrame(query, columns=['dt', 'god_cha', 'mes_cha', 'deg']) # преобразовали список в dataframe pandas
df['dt'] = pd.to_datetime(df['dt']) # преобразовали в формат даты и времени из текста
df['dt'] = df['dt'].dt.floor('h') # обнулили минуты и секунды

# Создание новой таблицы с часовыми интервалами
result = []

df.sort_values (by = ['dt'], ascending = [True], inplace = True)

for i in range(len(df) - 1):
    current_time = df.iloc[i]['dt']
    next_time = df.iloc[i + 1]['dt']
    j = False
    while current_time < next_time:
        if not j:
            result.append([current_time, df.iloc[i]['god_cha'], df.iloc[i]['mes_cha'], df.iloc[i]['deg'], df.iloc[i]['deg'] ])
        else:
            result.append([current_time, df.iloc[i]['god_cha'], df.iloc[i]['mes_cha'], None, df.iloc[i]['deg']])
        
        current_time += timedelta(hours=1)
        j = True

# Добавляем последний интервал до конца
last_feature_1 = df.iloc[-1]['god_cha']
last_feature_2 = df.iloc[-1]['mes_cha']
current_time = df.iloc[-1]['dt']
end_time = current_time + timedelta(days=30)  # Примерное добавление одного дня для последнего интервала
while current_time < end_time:
    result.append([current_time, last_feature_1, last_feature_2])
    current_time += timedelta(hours=1)

# Создание DataFrame из результатов
result_df = pd.DataFrame(result, columns=['dt','god_cha','mes_cha', 'deg', 'deg_for_season'])
result_df['god'] = result_df['dt'].dt.year
result_df['mes'] = result_df['dt'].dt.month
result_df['den'] = result_df['dt'].dt.day
result_df['chas'] = result_df['dt'].dt.hour

# Записываем в базу
conn = sqlite3.connect(V.bd_name)
cursor = conn.cursor()

# Запись результатов обратно в SQLite (в новую таблицу)
result_df.to_sql(tb_nm, conn, if_exists='replace', index=False)

# Закрытие соединения с базой данных
conn.close()

In [8]:
# Объединяем ветви года и месяца
tb_nm = 't_bazci_02'
sql = """
select vh3.*
, case
	when vh3.level_year_for_month = 1 then spr4.high_period
	when vh3.level_year_for_month = 2 then spr4.avg_period
	when vh3.level_year_for_month = 3 then spr4.low_period
  end as mes_num_rasklad

from(
select vh2.*
, spr2.jiazi_month as mes_heavenly_stems
, spr3.hieroglyph_cha as mes_earthly_branches
from (

select 
vh.dt
, vh.deg
, vh.deg_for_season
, vh.god	
, vh.mes	
, vh.den	
, vh.chas	
, vh.god_cha
, vh.mes_cha
, spr.num_situation as god_num_rasklad
, spr.heavenly_stems as god_heavenly_stems
, spr.earthly_branches as god_earthly_branches
, spr.level_year_for_month
from t_bazci_01 vh

-- небесная и земные ветви года
left join t_cyrcle_jiazi_for_year spr
on vh.god_cha = spr.year

where 1 = 1 
	) vh2

-- небесная ветвь месяца
left join t_spr_jiazi_for_month spr2
on vh2.god_heavenly_stems = spr2.jiazi_year 
and vh2.mes_cha = spr2.num_mont_cha

-- земная ветвь
left join t_spr_12_animal spr3
on vh2.mes_cha = spr3.id_animal
)vh3
--номер расклада месяца
left join t_spr_rasklad_for_month spr4
on vh3.mes_heavenly_stems = spr4.heavenly_stems
and vh3.mes_earthly_branches = spr4.earthly_branches
"""
MT.drop_table(tb_nm)
MT.create_table ( tb_target = tb_nm
                   , sql_script = sql                   
                  )

Таблица t_bazci_02 успешно удалена
Таблица t_bazci_02 Успешно создана
Скрипт create_table выполнился за 0 дней 0 часов 0 минут 0 секунд (с 2024-08-23 09:00:54 по 2024-08-23 09:00:54)


## Формируем справочник t_spr_rasklad_for_month

# Расчет цикла 60 Дзя Дзи
heavenly_stems = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛', '壬', '癸']
earthly_branches = ['子', '丑', '寅', '卯', '辰', '巳', '午', '未', '申', '酉', '戌', '亥']

# Функция для генерации всех комбинаций Дзя Дзи
def generate_jiazi():
    jiazi = []
    for i in range(60):
        stem = heavenly_stems[i % 10]
        branch = earthly_branches[i % 12]
        jiazi.append([stem, branch, stem + branch])
    return jiazi
#tb_nm = 't_spr_rasklad_for_month'
tb_nm = 't_spr_rasklad_for_hour'
# Вызываем функцию и выводим результат
all_jiazi = generate_jiazi()
for index, combination in enumerate(all_jiazi):
    MT.insert_data(tb = tb_nm, rezim = 'variable', column = ['heavenly_stems','earthly_branches','jazi'], value = combination)

## Определяем столп дня

In [9]:
tb_nm = 't_bazci_03'
def day_stem_branch(input_date):
    # Базовая дата: 31 января 1984 года, 甲子 (Jia Zi)
    base_date = date(1984, 1, 31)
    stems = ["甲", "乙", "丙", "丁", "戊", "己", "庚", "辛", "壬", "癸"]
    branches = ["子", "丑", "寅", "卯", "辰", "巳", "午", "未", "申", "酉", "戌", "亥"]

    # Вычисляем разницу в днях между базовой датой и вводимой датой
    delta_days = (input_date - base_date).days
    stem_index = (delta_days % 10)  # 10 небесных стволов
    branch_index = (delta_days % 12)  # 12 земных ветвей

    # Корректировка индексов для нулевого дня
    return stems[stem_index], branches[branch_index]


# Подключение к базе данных SQLite
conn = sqlite3.connect(V.bd_name)  # Замените 'your_database.db' на путь к вашей базе данных
cursor = conn.cursor()

# Загрузка данных из таблицы
query = "SELECT * FROM t_bazci_02"  # Замените 'date_column' и 'your_table' на реальные имена
df = pd.read_sql_query(query, conn, parse_dates=['dt'])

# Применение функции day_stem_branch к каждой дате в DataFrame
df['dt_date'] =df['dt'].dt.date
df['den_heavenly_stems'], df['den_earthly_branches'] = zip(*df['dt_date'].apply(day_stem_branch))

df['stolp_goda'] = df['god_heavenly_stems']+df['god_earthly_branches']
df['stolp_mes'] = df['mes_heavenly_stems']+df['mes_earthly_branches']
df['stolp_den'] = df['den_heavenly_stems']+df['den_earthly_branches']

# Вывод результата

# Запись результатов обратно в SQLite (в новую таблицу)
df.to_sql(tb_nm, conn, if_exists='replace', index=False)

# Закрытие соединения с базой данных
conn.close()

## Определеяем полярность дня (Инь или Ян) и номер расклада

In [10]:
tb_nm = 't_bazci_04'

# Подключение к базе данных SQLite
conn = sqlite3.connect(V.bd_name)  # Замените 'your_database.db' на путь к вашей базе данных
cursor = conn.cursor()

# Загрузка данных из таблицы
query = "SELECT dt, deg, deg_for_season, stolp_den FROM t_bazci_03 order by dt"  # Замените 'date_column' и 'your_table' на реальные имена
df = pd.read_sql_query(query, conn, parse_dates=['dt'])

# Закрытие соединения с базой данных
conn.close()

# Функция для определения янских и инских периодов
def classify_dates(df):
    df['year'] = df['dt'].dt.year
    periods = []

    for year in df['year'].unique():
        year_data = df[df['year'] == year]
        
        # Проверяем наличие необходимых значений deg
        if not (90 in year_data['deg'].values and 270 in year_data['deg'].values):
            print(f"Warning: Year {year} does not have required deg values.")
            continue
        
        # Находим даты для deg=90 и deg=270
        date_90 = year_data[year_data['deg'] == 90]['dt'].min()
        date_270 = year_data[year_data['deg'] == 270]['dt'].min()
        
        # Проверяем наличие дат с '甲子'
        if not ('甲子' in year_data['stolp_den'].values):
            print(f"Warning: Year {year} does not have any date with '甲子'.")
            continue
        
        # Находим максимально близкие даты с '甲子' до этих дат
        date_1 = year_data[(year_data['stolp_den'] == '甲子') & (date_90 - timedelta(days=59) <= year_data['dt']) & (year_data['dt'] <= date_90)]['dt'].min()
        date_2 = year_data[(year_data['stolp_den'] == '甲子') & (date_270 - timedelta(days=59) <= year_data['dt']) & (year_data['dt'] <= date_270)]['dt'].min()
         
        # Сохраняем периоды
        periods.append((date_1, date_2))

    # Присваиваем каждой дате её период
    def get_period(deg_for_season):
        """
        for start_date, end_date in periods:
            if start_date <= date < end_date and start_date.year == date.year == end_date.year:
                return 1
            elif (date < start_date or date  >= end_date) and  start_date.year == date.year == end_date.year:
                return 2
        """
        if deg_for_season in (315, 330, 345, 0, 15, 30, 45, 60, 75, 270, 285, 300) :
            return 2
        elif deg_for_season in (90, 105, 120, 135, 150, 165, 180, 195, 210, 225, 240, 255) :
            return 1
        return None

    df['period'] = df['deg_for_season'].apply(get_period)

    # считаем верхий, средний и нижний периоды для дня
    df = df.sort_values(by='dt')

    df['count_jiazi'] = 0 # Добавление колонки для подсчета встречаемости '甲子'
       
    count = 1 # Вспомогательная переменная для подсчета
    current_period = df.iloc[0]['period']
    current_day = df.iloc[0]['dt'].date()
    
    # Проходим по всем строкам DataFrame
    flag = False
    for index, row in df.iterrows():
        if row['period'] != current_period:
            # Сброс счетчика при смене периода
            count = 1
            flag = True
            current_period = row['period']
        
        if row['stolp_den'] == '甲子' and current_day != row['dt'].date():
            
            if flag:
                flag = False
                count = 1
            else:
                count += 1
            current_day = row['dt'].date()
        
        # Обновляем значение в новой колонке
        df.at[index, 'count_jiazi'] = count #if count else 1

    return df

# Вызываем функцию и выводим результаты
result_df = classify_dates(df)

# Подключение к базе данных SQLite
conn = sqlite3.connect(V.bd_name)  
cursor = conn.cursor()

result_df.to_sql(tb_nm, conn, if_exists='replace', index=False)

# Закрытие соединения с базой данных
conn.close()

In [11]:
# проставляем номер расклада для дня
tb_nm = 't_bazci_05'
sql= """
select vn.*
, spr.id as id_rasklad_day_from_spr
, case 
	when den_yin_yan = 1 and level_den = 1 then yin_high_period
	when den_yin_yan = 1 and level_den = 2 then yin_avg_period
	when den_yin_yan = 1 and level_den = 3 then yin_low_period
	
	when den_yin_yan = 2 and level_den = 1 then yan_high_period
	when den_yin_yan = 2 and level_den = 2 then yan_avg_period
	when den_yin_yan = 2 and level_den = 3 then yan_low_period
	
  end as den_num_rasklad

from (
select 
 vh.*
  , z.period as den_yin_yan
  , z.count_jiazi as level_den
  , s.id as id_season_start
from t_bazci_03 vh

-- добавили тип периода Инь/янь по дню
left join t_bazci_04 z
on vh.dt = z.dt 

-- добавляем id сезона стартовое по справочнику
left join t_spr_24_season s
on vh.deg_for_season = s.deg
) vn

left join t_spr_rasklad_for_day spr
on vn.stolp_den = spr.jazi
"""
MT.drop_table(tb_nm)
MT.create_table ( tb_target = tb_nm
                   , sql_script = sql                   
                  )

Таблица t_bazci_05 успешно удалена
Таблица t_bazci_05 Успешно создана
Скрипт create_table выполнился за 0 дней 0 часов 0 минут 0 секунд (с 2024-08-23 09:01:06 по 2024-08-23 09:01:07)


## Определяем столп и номер расклада для часа

In [12]:
tb_nm = 't_hour_jazi'
import datetime

def generate_branches(start_year, end_year):
    # Небесные стебли (иероглифы)
    celestial_stems = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛', '壬', '癸']
    # Земные ветви (иероглифы)
    earthly_branches = ['子', '丑', '寅', '卯', '辰', '巳', '午', '未', '申', '酉', '戌', '亥']

    base_date = datetime.datetime(1900, 1, 1)
    
    # Список для хранения результатов
    hour_pillars = []

    # Перебираем каждый год в заданном диапазоне
    for year in range(start_year, end_year + 1):
        # Перебираем каждый месяц
        for month in range(1, 13):
            # Перебираем каждый день месяца
            for day in range(1, 32):
                # Проверяем корректность даты (учитывая високосные годы и количество дней в месяце)
                try:
                    current_date = datetime.datetime(year, month, day)
                except ValueError:
                    continue
                
                days_difference = (current_date - base_date).days
                
                # Перебираем каждый час в дне
                for hour in range(0, 24):
                    # Определение индекса для земных и небесных ветвей
                    # Каждая ветвь охватывает 2 часа, начиная с 23:00 предыдущего дня
                    hour_index = (hour + 1) // 2 % 12
                    day_stem_index = (days_difference * 2 + hour_index) % 10
                    
                    
                    # Формирование столпа часа
                    hour_pillar = f"{celestial_stems[day_stem_index]}-{earthly_branches[hour_index]}"
                    hour_pillars.append([current_date.year, current_date.month, current_date.day, hour
                                         , celestial_stems[day_stem_index], earthly_branches[hour_index]
                                        , f"{celestial_stems[day_stem_index]}{earthly_branches[hour_index]}"])

    return hour_pillars

# Пример использования функции
hour_pillars = generate_branches(year_st, year_en)

df = pd.DataFrame(hour_pillars, columns=['god','mes','den','chas','chas_heavenly_stems','chas_earthly_branches','stolp_chas'])
# Подключение к базе данных SQLite
conn = sqlite3.connect(V.bd_name)  
cursor = conn.cursor()

df.to_sql(tb_nm, conn, if_exists='replace', index=False)

# Закрытие соединения с базой данных
conn.close()

In [13]:
tb_nm = 't_bazci_06'
sql= """
Select 
vh.*
, h.chas_heavenly_stems
, h.chas_earthly_branches
, h.stolp_chas
from t_bazci_05 vh

left join t_hour_jazi h
on vh.god = h.god
and vh.mes = h.mes
and vh.den = h.den
and vh.chas = h.chas
"""

MT.drop_table(tb_nm)
MT.create_table(tb_target = tb_nm
                   , sql_script = sql)

Таблица t_bazci_06 успешно удалена
Таблица t_bazci_06 Успешно создана
Скрипт create_table выполнился за 0 дней 0 часов 0 минут 1 секунд (с 2024-08-23 09:01:09 по 2024-08-23 09:01:10)


In [14]:
tb_nm = 't_bazci_07'

# определяем палярность и номер расклада
sql = """
select dt, id_season_start, den_heavenly_stems, den_yin_yan 
from t_bazci_06
"""
query = MT.gp_execute(sql_script = sql, rezim = 'list')
df = pd.DataFrame(query, columns=['dt', 'id_season_start', 'den_heavenly_stems', 'den_yin_yan']) # преобразовали список в dataframe pandas
df['dt'] = pd.to_datetime(df['dt'])
df['period_hour'] = 0 # добавили колонку для оценки периода часа
df['id_season_for_hour'] = 0 # добавили колонку для сезона часа

#current_season = df.iloc[0]['den_heavenly_stems']
current_day = df.iloc[0]['dt'].date()

current_count_period = 1
current_id_season = 20
start_dt = date(year = 2023, month = 11, day = 22)
# Проходим по всем строкам DataFrame
Flag = False
for index, row in df.iterrows():
    if row['dt'].date() >= start_dt:
        if row['den_heavenly_stems'] in ['甲', '己'] and current_day != row['dt'].date():
                 
            if current_count_period < 3:
                current_count_period +=1
            elif current_count_period == 3:
    
                current_count_period = 1
            
                if  ((current_day.year in (1990, 2024) and current_id_season == 21) or (current_day.year in (2030, 2033, 2036, 2039, 2042) and current_id_season == 9)) and not Flag:
                        current_id_season-=1
                        Flag = True
                    
                if  current_id_season <= 23:
                    current_id_season +=1
                    if current_id_season in (22, 10) and current_day.year in (2024, 2030, 2033, 2036, 2039, 2042):
                        Flag = False
                else: 
                    current_id_season = 1
            current_day = row['dt'].date()
        # Обновляем значение в новой колонке
        df.at[index, 'period_hour'] = current_count_period
        df.at[index, 'id_season_for_hour'] = current_id_season

# Подключение к базе данных SQLite
conn = sqlite3.connect(V.bd_name)  
cursor = conn.cursor()

df.to_sql(tb_nm, conn, if_exists='replace', index=False)

# Закрытие соединения с базой данных
conn.close()

In [15]:
tb_nm = 't_bazci'
sql= """
select 
  vn.dt
, vn.deg
, vn.deg_for_season
, vn.god
, vn.mes
, vn.den
, vn.chas
, vn.god_cha
, vn.mes_cha
, vn.id_season_start

, vn.stolp_goda
, vn.stolp_mes
, vn.stolp_den
, vn.stolp_chas

, vn.god_num_rasklad
, vn.mes_num_rasklad

, vn.id_rasklad_day_from_spr
, vn.level_den
, vn.den_yin_yan
, vn.den_num_rasklad

, vn.period_hour
, vn.yin_yan as chas_yin_yan
, vn.num_rasklad as  chas_num_rasklad

, vn.god_heavenly_stems
, vn.god_earthly_branches
, vn.mes_heavenly_stems
, vn.mes_earthly_branches
, vn.den_heavenly_stems
, vn.den_earthly_branches
, vn.chas_heavenly_stems
, vn.chas_earthly_branches

from (
select 
	vh.*
	, t2.period_hour  
	, spr.yin_yan
	, case 
		when t2.period_hour = 1 then spr.high_period
		when t2.period_hour = 2 then spr.avg_period
		when t2.period_hour = 3 then spr.low_period
	end as num_rasklad
	
from t_bazci_06 vh

left join t_bazci_07 t2
on vh.dt = t2.dt

left join t_spr_24_season spr
on t2.id_season_for_hour =spr.id
) vn

"""

MT.drop_table(tb_nm)
MT.create_table(tb_target = tb_nm
                   , sql_script = sql)

Таблица t_bazci успешно удалена
Таблица t_bazci Успешно создана
Скрипт create_table выполнился за 0 дней 0 часов 0 минут 0 секунд (с 2024-08-23 09:01:19 по 2024-08-23 09:01:19)


In [8]:
from IPython.display import Javascript, HTML, display
import json
import os
import nbformat
from nbconvert import PythonExporter

def get_notebook_information():
    """
    Получает имя текущего блокнота и его путь с использованием JavaScript.
    """
    js = """
    require(["base/js/namespace"], function(Jupyter) {
        var notebook_name = Jupyter.notebook.notebook_name;
        var notebook_path = Jupyter.notebook.notebook_path;
        var data = {
            "notebook_name": notebook_name,
            "notebook_path": notebook_path
        };
        var kernel = IPython.notebook.kernel;
        kernel.execute("NOTEBOOK_INFO = " + JSON.stringify(data));
    });
    """
    display(Javascript(js))

get_notebook_information()

try:
    notebook_filename = NOTEBOOK_INFO['notebook_name']
    if notebook_filename == '':
        print("Не удалось получить имя файла блокнота.")
        notebook_filename = 'default_notebook.ipynb'
    else:
        print(f"Имя файла блокнота: {notebook_filename}")
except NameError:
    print("Не удалось получить имя файла блокнота через JavaScript.")
    notebook_filename = 'default_notebook.ipynb'

output_filename = notebook_filename.replace('.ipynb', '.py')

try:
    with open(notebook_filename, 'r', encoding='utf-8') as f:
        notebook_content = nbformat.read(f, as_version=4)

    python_exporter = PythonExporter()
    (body, resources) = python_exporter.from_notebook_node(notebook_content)

    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write(body)
    print(f'Файл успешно сохранен в {output_filename}')

except Exception as e:
    print(f'Произошла ошибка: {e}')


<IPython.core.display.Javascript object>

Не удалось получить имя файла блокнота через JavaScript.
Произошла ошибка: [Errno 2] No such file or directory: 'default_notebook.ipynb'


In [1]:
import nbformat
from nbconvert import PythonExporter

# Путь к текущему блокноту
nm = 'bazci 03'
notebook_path = f'{nm}.ipynb'  # ← заменить на имя твоего файла

# Чтение блокнота
with open(notebook_path, 'r', encoding='utf-8') as f:
    notebook = nbformat.read(f, as_version=4)

# Экспорт в .py
exporter = PythonExporter()
script, _ = exporter.from_notebook_node(notebook)

# Сохранение
output_file = f'{nm}.py'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(script)

print(f"Файл успешно сохранён как {output_file}")

Файл успешно сохранён как bazci 03.py
